# 📝 과제 02: 날짜로 요일 맞히기

## 만들 결과

`연`, `월`, `일`만 입력으로 받아 요일을 맞히는 DNN을, 모델 구조와 학습 설정은 같게 두고 날짜를 숫자 3칸과 원-핫 73칸으로 달리 넣어 두 번 학습합니다. 실무 데이터에도 매장 코드, 지역 번호처럼 숫자로 적힌 열이 많아, 이런 열을 숫자 그대로 넣을지 원-핫으로 펼칠지에 따라 결과가 얼마나 달라지는지 확인합니다.

<img src="images/41_date_onehot.png" width="1050" alt="제목은 '날짜를 숫자 3개 대신 칸 73개로 나타내기'입니다. 위쪽 노란 상자에 '2024년 10월 20일'이 적혀 있습니다. 왼쪽 파란 판 '숫자 입력(3칸)'에는 '연', '월', '일' 열에 '2024', '10', '20'이 든 표가 있습니다. 화살표 오른쪽 초록 판 '원-핫 입력(73칸)'에는 줄 세 개가 있습니다. '연 30칸(2000~2029)' 줄은 '2024' 칸만 주황색 1이고, '월 12칸(1~12)' 줄은 '10' 칸만 주황색 1이고, '일 31칸(1~31)' 줄은 '20' 칸만 주황색 1이며 나머지 칸은 모두 0입니다.">

- 숫자 입력(3칸)의 학습 기록: 20에폭 뒤 테스트 정확도 약 0.115로, 무작위로 고를 때의 정확도(약 0.14)와 같은 수준입니다. 학습률을 바꿔도 같습니다.
- 원-핫 입력(73칸)의 학습 기록: 20에폭 뒤 테스트 정확도 약 0.99입니다.
- 옵티마이저와 학습률 여섯 설정의 학습 손실 곡선과, 규칙으로 고른 설정(Adam 0.003).
- 2024년까지 학습하고 2025~2029년을 맞힌 결과: 학습 데이터 정확도 1.00, 테스트 정확도 약 0.10.

## 준비: 데이터와 제공 코드

- **자료:** `data/과제02/dates.csv`(10,958행). 2000-01-01부터 2029-12-31까지 하루가 한 행이고, 열은 `연`, `월`, `일`, `요일`입니다. `요일`은 0=월, 1=화, 2=수, 3=목, 4=금, 5=토, 6=일입니다.
- **제공 코드:** 준비 셀이 아래 이름과 교안 02의 `train_one_epoch`, `evaluate`를 만듭니다.

| 이름 | 내용 |
|---|---|
| `dates` | 달력 표(10,958행) |
| `y_all` | 모든 행의 요일(정수 텐서) |
| `train_index`, `test_index` | 무작위로 섞어 나눈 학습 행 번호 8,766개, 테스트 행 번호 2,192개 |
| `y_train`, `y_test` | 학습 행과 테스트 행의 요일 |

- **작성:** `[작성]` 셀의 `...`를 채우고 바로 아래 `[자가채점]` 셀을 실행하세요. 뒤 문항이 앞 문항의 변수를 쓰므로 차례로 풉니다.
- **무작위로 고를 때의 정확도:** 요일이 7개라 아무것도 배우지 못한 모델은 정확도 1/7(약 0.14) 안팎, 손실 $\log 7 \approx 1.95$ 근처입니다. 결과를 이 두 값과 비교하세요.
- **로짓(logit):** 출력층이 칸마다 내는 값입니다. softmax를 거치면 확률이 됩니다. 이 과제의 출력층은 요일 7칸의 로짓 7개를 냅니다.
- **실행 시간:** CPU에서 과제 전체가 1분 안팎입니다.

In [ ]:
import platform
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
# 2000~2029년 달력 10,958일
dates = pd.read_csv(data_dir / "과제02" / "dates.csv")
print("행 수:", len(dates))

# 첫 일주일: 요일이 5(토)에서 시작해 0~6을 한 번 돕니다
first_week = dates.head(7)
display(first_week)

In [ ]:
# 정답: 모든 행의 요일. CrossEntropyLoss는 정답을 정답 칸 번호(정수, torch.long)로 받습니다
y_all = torch.tensor(dates["요일"].to_numpy(), dtype=torch.long)

# 행 번호를 무작위로 섞어 80%는 학습, 20%는 테스트에 씁니다(시드 0으로 고정)
generator = torch.Generator().manual_seed(0)
order = torch.randperm(len(dates), generator=generator)
n_train = int(len(dates) * 0.8)
train_index = order[:n_train]
test_index = order[n_train:]
y_train = y_all[train_index]
y_test = y_all[test_index]
print("학습 행:", len(train_index), "| 테스트 행:", len(test_index))

In [ ]:
# 교안 02에서 만든 학습, 평가 함수입니다. 실행만 하고 넘어갑니다.
def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None):
    """미니배치마다 파라미터를 한 번씩 갱신하고, 그 에폭 미니배치 손실의 평균을 돌려줍니다."""
    total_loss = 0.0
    for X_batch, y_batch in loader:
        # 미니배치를 모델이 있는 계산 장치로 옮깁니다
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # 지난 스텝의 기울기를 지우고, 순전파 -> 손실 -> 역전파 -> 갱신
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

        # 스케줄러가 있으면 스텝마다 학습률을 바꿉니다
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()

    # len(loader)는 미니배치 수입니다
    return total_loss / len(loader)


def evaluate(model, loader, loss_fn):
    """기울기 없이 예측해 (평균 손실, 정확도)를 돌려줍니다."""
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(X_batch)

            # 마지막 미니배치는 장 수가 적으므로, 손실에 장 수를 곱해 더한 뒤 전체 장 수로 나눕니다
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()

    n_samples = len(loader.dataset)
    return total_loss / n_samples, correct / n_samples

## 1. 날짜 숫자를 그대로 넣어 학습하기

먼저 `연`, `월`, `일` 숫자 세 개를 그대로 입력으로 씁니다.

### 1-1. 숫자 입력 만들기

**배경**: `연`(2000~2029)과 `일`(1~31)은 값의 크기가 크게 달라 표준화해서 넣습니다.

**요구사항**
- `연`, `월`, `일` 세 열을 float32 텐서 **`X_number_raw`**(모양 `(10958, 3)`)로 만드세요.
- 학습 행(`train_index`)만으로 계산한 열 평균을 **`train_mean`**(모양 `(3,)`)에 담으세요.
- 학습 행(`train_index`)만으로 계산한 열 표준편차를 **`train_std`**(모양 `(3,)`)에 담으세요. 표준편차는 교안 01처럼 `.std()` 기본값으로 구합니다.
- `X_number_raw`를 `train_mean`, `train_std`로 표준화해 **`X_number`**(float32, 모양 `(10958, 3)`)에 담으세요.
- `X_number`에서 학습 행만 고른 텐서를 **`X_number_train`** 에 담으세요.
- `X_number`에서 테스트 행만 고른 텐서를 **`X_number_test`** 에 담으세요.

**예시**: 첫 행(2000년 1월 1일)은 가장 이른 연도, 1월, 1일이라 세 값이 모두 평균보다 작은 -1.6 안팎이 됩니다.

**확인 기준**: `X_number`는 `(10958, 3)` 모양의 float32 텐서이고, 학습 행만 보면 열마다 평균 0, 표준편차 1입니다. `train_mean`, `train_std`의 모양이 `(3,)`이고, `X_number_train`, `X_number_test`는 `X_number`에서 각각 `train_index`, `test_index` 행을 고른 것입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 평균과 표준편차는 학습 행에서만 구하고, 그 값으로 전체 행을 표준화합니다.

세부구현:
1. dates에서 연, 월, 일 열을 골라 넘파이 배열로 바꾼 뒤 텐서로 만들면서 자료형을 float32로 지정합니다.
2. 1의 텐서에서 train_index 행만 고른 뒤, 열마다 하나씩 구하도록 0번 축 방향으로 평균과 표준편차를 구합니다.
3. 전체 텐서에서 평균을 빼고 표준편차로 나눕니다.
4. X_number에서 train_index 행과 test_index 행을 각각 고릅니다.
```

</details>

In [ ]:
# [작성] 연, 월, 일 숫자를 표준화한 입력을 만드세요.

# 1. 연, 월, 일 세 열을 float32 텐서로
X_number_raw = ...

# 2. 학습 행의 열 평균과 표준편차로 표준화
train_mean = ...
train_std = ...
X_number = ...

# 3. 학습 입력과 테스트 입력
X_number_train = ...
X_number_test = ...
print("첫 행(2000-01-01):", X_number[0])

In [ ]:
# [자가채점] 입력 모양과 표준화 기준을 확인합니다.
assert isinstance(X_number, torch.Tensor) and X_number.shape == (len(dates), 3), "X_number는 행마다 연, 월, 일 세 값이 든 (10958, 3) 텐서입니다. `...`를 채웠는지 확인하세요."
assert X_number.dtype == torch.float32, "X_number는 float32 텐서로 만드세요. 텐서를 만들 때 자료형을 float32로 지정합니다."
assert train_mean.shape == (3,) and train_std.shape == (3,), "train_mean, train_std는 열마다 하나씩, 모양 (3,)입니다. 0번 축 방향으로 계산하세요."

train_mean_check = X_number[train_index].mean(dim=0)
train_std_check = X_number[train_index].std(dim=0)
assert (train_mean_check.abs() < 1e-3).all(), f"학습 행의 열 평균이 0이 아닙니다({train_mean_check.tolist()}). 평균과 표준편차를 전체 행이나 테스트 행이 아니라 학습 행(train_index)에서 구했는지 보세요."
assert ((train_std_check - 1).abs() < 1e-3).all(), f"학습 행의 열 표준편차가 1이 아닙니다({train_std_check.tolist()}). 학습 행의 표준편차로 나눴는지 보세요."
assert torch.equal(X_number_train, X_number[train_index]), "X_number_train은 X_number에서 train_index 행을 고른 것입니다."
assert torch.equal(X_number_test, X_number[test_index]), "X_number_test는 X_number에서 test_index 행을 고른 것입니다."
print("1-1 확인 완료")

### 1-2. 요일 분류 DNN 학습 함수 만들기

**배경**: 입력과 학습 설정만 바꿔 같은 DNN을 여러 번 학습하므로, 교안 02의 학습 과정을 함수 하나로 묶습니다.

**요구사항**
- 골격의 `...` 아홉 곳을 채워 **`train_weekday_model(X_train, y_train, X_test, y_test, lr, epochs, optimizer_name)`** 을 완성하세요. 이 함수는 (모델, 기록 표 DataFrame)을 돌려줍니다.
- 미니배치: 학습용, 테스트용 모두 배치 크기 64이고 학습용만 섞습니다.
- 모델: `nn.Sequential`로 `입력 열 수 -> 은닉층 뉴런 128개 -> ReLU -> 은닉층 뉴런 128개 -> ReLU -> 출력 7`을 만들어 `device`로 옮깁니다. 입력 열 수는 `X_train`의 열 수입니다(숫자 입력이면 3).
- 옵티마이저: `"sgd"`는 채워 두었습니다. `"momentum"`은 모멘텀 계수 0.9인 `torch.optim.SGD`, 그 밖(`"adam"`)은 `torch.optim.Adam`이며 둘 다 학습률은 `lr`입니다.
- 학습 손실: 에폭마다 `train_one_epoch`가 돌려준 값입니다.
- 학습 데이터 정확도: 에폭마다 학습 데이터 전체를 한 번에 모델에 넣어 예측한 결과로 계산합니다(교안 02 5-4와 같은 방법). 섞는 학습용 로더를 다시 돌리면 다음 에폭의 미니배치 순서가 달라집니다.
- 테스트 정확도: 에폭마다 `evaluate`를 테스트용 로더로 불러 구합니다.

**예시**: `train_weekday_model(X_number_train, y_train, X_number_test, y_test, lr=0.001, epochs=2, optimizer_name="adam")`은 (모델, 기록 표)를 돌려주고, 기록 표는 아래 모양입니다(값은 예).

| 에폭 | 학습 손실 | 학습 데이터 정확도 | 테스트 정확도 |
|---|---|---|---|
| 1 | 1.95 | 0.14 | 0.13 |
| 2 | 1.95 | 0.15 | 0.12 |

**확인 기준**: 기록 표의 열이 `에폭`, `학습 손실`, `학습 데이터 정확도`, `테스트 정확도` 순서이고 행 수가 에폭 수와 같으며, 모델이 위 층 구성이고 마지막 에폭의 테스트 정확도가 0~1입니다. `"sgd"`, `"momentum"`, `"adam"`마다 학습용 DataLoader가 배치 크기 64로 학습 데이터를 섞고, 옵티마이저가 지문의 종류, 학습률 lr, 모멘텀 계수(momentum은 0.9)로 만들어집니다. 에폭마다 학습용 DataLoader와 테스트용 DataLoader를 한 번씩만 돌리며(학습 데이터 정확도는 학습 데이터 전체를 한 번에 모델에 넣어 구함), 기록 표의 마지막 에폭 정확도는 그 에폭을 마친 모델의 정확도와 거의 같고(학습 데이터 정확도 0.002, 테스트 정확도 0.02 이내), 같은 입력으로 두 번 학습하면 같은 모델이 나옵니다(시드 고정). 숫자 입력 640행의 1에폭 결과는 손실이 log(7) = 1.946 근처이고 학습 데이터 정확도가 무작위 수준(약 1/7)입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 미니배치 학습 루프를 함수 안에 옮깁니다. 바뀌는 값(입력, 학습률, 에폭 수, 옵티마이저 이름)은 인자로 받습니다.

세부구현:
1. 입력과 정답을 데이터셋으로 묶어 DataLoader에 넘기고, 학습용만 섞습니다.
2. 첫 선형 층의 입력 크기는 X_train의 열 수(shape의 둘째 값)이고, 마지막 선형 층의 출력 크기는 요일 수 7입니다.
3. 모멘텀과 Adam도 sgd 줄처럼 학습할 파라미터와 학습률을 넘기고, 모멘텀은 계수를 하나 더 줍니다.
4. train_one_epoch가 돌려주는 값이 학습 손실입니다.
5. 학습 데이터 정확도는 기울기 계산 없이 학습 데이터 전체를 모델에 넣고, 로짓이 가장 큰 칸의 번호가 정답과 같은 비율로 구합니다. 계산 장치에 있는 예측은 CPU로 옮겨 정답과 비교합니다.
6. evaluate는 (평균 손실, 정확도) 두 값을 돌려주므로 둘째 값이 테스트 정확도입니다.
```

</details>

In [ ]:
# [작성] 요일 분류 DNN을 학습하는 함수를 완성하세요.
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


def train_weekday_model(X_train, y_train, X_test, y_test, lr, epochs, optimizer_name):
    """요일 분류 DNN을 학습하고 (모델, 에폭별 기록 표)를 돌려줍니다."""
    # 1. 모델 초깃값과 미니배치 섞는 순서가 실행마다 같게 시드 고정
    torch.manual_seed(0)

    # 2. 학습용(섞기), 테스트용(섞지 않기) 미니배치
    train_loader = ...
    test_loader = ...

    # 3. 은닉층 두 개인 DNN과 손실함수
    model = ...

    # 교차 엔트로피 = 평균(-log(정답 칸의 확률)), 확률은 로짓 7개에 softmax를 적용한 값(손실 안에서 계산)
    loss_fn = nn.CrossEntropyLoss()

    # 4. optimizer_name에 맞는 옵티마이저
    if optimizer_name == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    elif optimizer_name == "momentum":
        optimizer = ...
    else:
        optimizer = ...

    # 5. 에폭마다 학습하고 학습 데이터, 테스트 데이터의 정확도를 기록
    rows = []
    for epoch in range(1, epochs + 1):
        train_loss = ...

        # 학습 데이터 전체를 한 번에 넣어 예측하고 정확도를 구합니다(섞는 train_loader를 다시 돌리지 않습니다)
        with torch.no_grad():
            train_predictions = ...
        train_accuracy = ...

        # 테스트 정확도
        _, test_accuracy = ...
        rows.append({"에폭": epoch, "학습 손실": train_loss, "학습 데이터 정확도": train_accuracy, "테스트 정확도": test_accuracy})

    # 6. (모델, 기록 표)
    return model, pd.DataFrame(rows)

In [ ]:
# [자가채점] 학습 함수가 제공 함수에 넘기는 설정과 기록 표, 모델 모양을 확인합니다.
import torch.utils.data as torch_data

# 학습 함수 안에서 train_one_epoch를 부를 때 넘긴 DataLoader, 옵티마이저 설정과 DataLoader를 돌린 횟수를 기록합니다
real_train_one_epoch = train_one_epoch
real_loader_iter = torch_data.DataLoader.__iter__
epoch_calls = []
loader_runs = []


def recording_train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None):
    group = optimizer.param_groups[0]
    epoch_calls.append({"배치 크기": loader.batch_size, "섞기": isinstance(loader.sampler, torch_data.RandomSampler), "옵티마이저": type(optimizer).__name__, "학습률": group["lr"], "모멘텀": group.get("momentum", 0)})
    return real_train_one_epoch(model, loader, loss_fn, optimizer, scheduler)


def recording_loader_iter(self):
    loader_runs.append((len(self.dataset), isinstance(self.sampler, torch_data.RandomSampler)))
    return real_loader_iter(self)


# 숫자 입력 640행으로 2에폭 학습해 기록 표와 모델 모양을 확인
check_X_train = X_number_train[:640]
check_y_train = y_train[:640]
check_X_test = X_number_test[:64]
check_y_test = y_test[:64]
run_calls = {}
run_histories = {}
run_models = {}
try:
    train_one_epoch = recording_train_one_epoch
    torch_data.DataLoader.__iter__ = recording_loader_iter
    check_model, check_history = train_weekday_model(check_X_train, check_y_train, check_X_test, check_y_test, lr=0.1, epochs=2, optimizer_name="sgd")
    two_epoch_runs = list(loader_runs)

    # 옵티마이저마다 같은 데이터로 1에폭 학습
    for check_name, check_lr in [("sgd", 0.1), ("momentum", 0.1), ("adam", 0.01)]:
        epoch_calls.clear()
        run_models[check_name], run_histories[check_name] = train_weekday_model(check_X_train, check_y_train, check_X_test, check_y_test, lr=check_lr, epochs=1, optimizer_name=check_name)
        run_calls[check_name] = list(epoch_calls)
finally:
    train_one_epoch = real_train_one_epoch
    torch_data.DataLoader.__iter__ = real_loader_iter

assert isinstance(check_history, pd.DataFrame), "train_weekday_model은 (모델, 기록 표)를 돌려줍니다. `...`를 채웠는지 확인하세요."
assert list(check_history.columns) == ["에폭", "학습 손실", "학습 데이터 정확도", "테스트 정확도"], "기록 표의 열은 에폭, 학습 손실, 학습 데이터 정확도, 테스트 정확도 순서입니다."
assert list(check_history["에폭"]) == [1, 2], "epochs=2이면 기록 표는 에폭 1, 2 두 행입니다."
assert isinstance(check_model, nn.Sequential), "모델은 nn.Sequential로 만드세요."
assert check_model[0].in_features == 3, "첫 선형 층의 입력 크기는 X_train의 열 수입니다. 숫자 입력은 3칸입니다."
assert check_model[-1].out_features == 7, "마지막 선형 층의 출력 크기는 요일 수 7입니다."
assert len(check_model) == 5, "모델은 선형 층, ReLU, 선형 층, ReLU, 선형 층 다섯 층입니다. 입력 -> 은닉층 뉴런 128개 -> ReLU -> 은닉층 뉴런 128개 -> ReLU -> 출력 7 구조인지 보세요."
assert check_model[2].in_features == 128 and check_model[2].out_features == 128, "가운데 선형 층은 128 -> 128입니다. 입력 -> 은닉층 뉴런 128개 -> ReLU -> 은닉층 뉴런 128개 -> ReLU -> 출력 7 구조인지 보세요."
assert 0 <= check_history["테스트 정확도"].iloc[-1] <= 1, "정확도는 0~1 사이입니다. evaluate가 돌려주는 두 값 가운데 둘째 값을 쓰세요."

# DataLoader는 에폭마다 학습용 한 번(train_one_epoch), 테스트용 한 번(evaluate)만 돌아야 합니다
train_runs = [shuffled for size, shuffled in two_epoch_runs if size == len(check_X_train)]
test_runs = [shuffled for size, shuffled in two_epoch_runs if size == len(check_X_test)]
assert len(train_runs) == 2, f"에폭 2번 동안 학습용 DataLoader를 {len(train_runs)}번 돌렸습니다. 에폭마다 train_one_epoch 안에서 한 번씩만 돌아야 합니다. 학습 데이터 정확도는 학습 데이터 전체를 한 번에 모델에 넣어 구하세요. DataLoader를 돌릴 때마다 난수가 쓰여 다음 에폭의 미니배치 순서가 달라집니다."
assert len(test_runs) == 2, f"에폭 2번 동안 테스트용 DataLoader를 {len(test_runs)}번 돌렸습니다. 에폭마다 evaluate로 한 번씩만 돌려 테스트 정확도를 구하세요."
assert all(train_runs), "학습용 DataLoader는 학습 데이터를 섞어야 합니다."
assert not any(test_runs), "테스트용 DataLoader는 섞지 않습니다."

# 제공 함수 train_one_epoch에 넘긴 미니배치와 옵티마이저 설정
expected_optimizers = {"sgd": ("SGD", 0), "momentum": ("SGD", 0.9), "adam": ("Adam", None)}
for check_name, check_lr in [("sgd", 0.1), ("momentum", 0.1), ("adam", 0.01)]:
    assert len(run_calls[check_name]) == 1, f"\"{check_name}\"으로 1에폭 학습하는 동안 train_one_epoch를 {len(run_calls[check_name])}번 불렀습니다. 에폭마다 한 번씩 부르세요."
    call = run_calls[check_name][0]
    class_name, momentum = expected_optimizers[check_name]
    assert call["배치 크기"] == 64 and call["섞기"], f"학습용 DataLoader는 배치 크기 64로 학습 데이터를 섞어야 합니다(지금 배치 크기 {call['배치 크기']}, 섞기 {call['섞기']})."
    assert call["옵티마이저"] == class_name, f"\"{check_name}\"이면 {class_name} 옵티마이저를 써야 합니다(지금 {call['옵티마이저']})."
    assert abs(call["학습률"] - check_lr) < 1e-9, f"\"{check_name}\"의 옵티마이저 학습률이 {call['학습률']}입니다. 인자로 받은 lr({check_lr})을 옵티마이저에 주세요. 학습률을 빼면 기본값으로 학습합니다."
    if momentum is not None:
        assert abs(call["모멘텀"] - momentum) < 1e-9, f"\"{check_name}\"의 모멘텀 계수가 {call['모멘텀']}입니다. {'모멘텀 계수 0.9를 주세요.' if momentum else 'sgd에는 모멘텀 계수를 주지 않습니다.'}"

    # 숫자 입력은 맞힐 수 있는 규칙이 없어 1에폭 뒤에도 손실이 log(7)=1.946 근처, 정확도가 무작위 수준(약 1/7)입니다
    first_row = run_histories[check_name].iloc[0]
    assert abs(first_row["학습 손실"] - 1.946) < 0.1, f"\"{check_name}\"(학습률 {check_lr})로 1에폭 학습한 손실이 {first_row['학습 손실']:.3f}입니다. 숫자 입력으로 1에폭 학습하면 log(7) = 1.946 근처입니다. 학습 손실을 train_one_epoch가 돌려준 값으로 기록했는지 보세요."
    assert abs(first_row["학습 데이터 정확도"] - 1 / 7) < 0.1, f"\"{check_name}\"의 학습 데이터 정확도가 {first_row['학습 데이터 정확도']:.3f}입니다. 무작위 수준(약 0.14)이어야 합니다. 맞힌 개수를 전체 개수로 나눴는지 보세요."

# 기록 표의 마지막 에폭 정확도는 그 에폭을 마친 모델의 정확도와 같아야 합니다
reference_loss_fn = nn.CrossEntropyLoss()
for finished_name, finished_model, finished_history in [("2에폭 sgd", check_model, check_history)] + [(f"1에폭 {name}", run_models[name], run_histories[name]) for name in run_models]:
    _, reference_train_accuracy = evaluate(finished_model, torch_data.DataLoader(torch_data.TensorDataset(check_X_train, check_y_train), batch_size=len(check_X_train)), reference_loss_fn)
    _, reference_test_accuracy = evaluate(finished_model, torch_data.DataLoader(torch_data.TensorDataset(check_X_test, check_y_test), batch_size=len(check_X_test)), reference_loss_fn)
    recorded_train_accuracy = finished_history["학습 데이터 정확도"].iloc[-1]
    recorded_test_accuracy = finished_history["테스트 정확도"].iloc[-1]
    assert abs(recorded_train_accuracy - reference_train_accuracy) < 0.002, f"{finished_name}: 마지막 에폭의 학습 데이터 정확도({recorded_train_accuracy:.3f})가 학습을 마친 모델로 학습 데이터 전체를 예측한 정확도({reference_train_accuracy:.3f})와 다릅니다. 그 에폭의 학습이 끝난 뒤에, 학습 데이터 전체로 구했는지 보세요."
    assert abs(recorded_test_accuracy - reference_test_accuracy) < 0.02, f"{finished_name}: 마지막 에폭의 테스트 정확도({recorded_test_accuracy:.3f})가 학습을 마친 모델의 테스트 정확도({reference_test_accuracy:.3f})와 다릅니다. 그 에폭의 학습이 끝난 뒤에 evaluate로 구했는지 보세요."

# 같은 입력으로 다시 학습하면 같은 모델이 나와야 합니다(시드를 고정하지 않으면 초깃값과 미니배치 순서가 달라집니다)
repeat_model, _ = train_weekday_model(check_X_train, check_y_train, check_X_test, check_y_test, lr=0.1, epochs=1, optimizer_name="sgd")
assert torch.allclose(run_models["sgd"][0].weight.detach().cpu(), repeat_model[0].weight.detach().cpu(), atol=1e-3), "같은 입력으로 두 번 학습했더니 모델이 달라졌습니다. 함수 맨 앞에서 시드를 고정했는지 보세요."
print("1-2 확인 완료")

### 1-3. 숫자 입력으로 학습하기

**배경**: 숫자 입력으로 학습한 결과가 뒤에서 원-핫 입력과 비교할 기준이 됩니다.

**요구사항**
- 숫자 입력(`X_number_train`, `X_number_test`)을 학습률 0.001, 20에폭, `"adam"`으로 학습하세요.
- 학습한 모델을 **`number_model`**(nn.Sequential)에 담으세요.
- 기록 표를 **`number_history`**(DataFrame)에 담으세요.

**확인 기준**: `number_history`는 20행이고 열은 `train_weekday_model`이 돌려주는 기록 표와 같으며, `number_model`의 첫 선형 층 입력 크기가 3입니다. 마지막 에폭은 무작위로 고를 때의 수준으로, 학습 손실이 1.8을 넘고 학습 데이터 정확도가 0.25 미만이며 테스트 정확도가 0.05~0.25입니다.

<details><summary>힌트</summary>

```text
접근방법:
- train_weekday_model에 숫자 입력 네 텐서와 학습 설정을 넘깁니다.

세부구현:
1. 입력은 X_number_train, X_number_test이고 정답은 준비 셀의 y_train, y_test입니다.
2. 학습률, 에폭 수, 옵티마이저 이름은 함수의 lr, epochs, optimizer_name 인자로 넘깁니다.
3. 함수가 (모델, 기록 표) 두 값을 돌려주므로 쉼표로 두 이름에 나눠 담습니다.
```

</details>

In [ ]:
# [작성] 숫자 입력으로 20에폭 학습하세요.
number_model, number_history = ...
display(number_history.round(3))

In [ ]:
# [자가채점] 기록 표 모양과 학습 결과를 확인합니다.
assert isinstance(number_history, pd.DataFrame) and len(number_history) == 20, "number_history는 20에폭의 기록 표(20행)입니다. `...`를 채웠는지 확인하세요."
assert number_model[0].in_features == 3, "숫자 입력이면 첫 선형 층의 입력 크기가 3입니다. X_number_train을 넘겼는지 보세요."

number_last = number_history.iloc[-1]
assert number_last["학습 손실"] > 1.8, f"마지막 학습 손실이 {number_last['학습 손실']:.3f}로 지문대로 학습한 결과와 다릅니다. 숫자 입력으로 학습률 0.001, 20에폭, adam을 넘겼는지 보세요."
assert number_last["학습 데이터 정확도"] < 0.25, f"마지막 학습 데이터 정확도가 {number_last['학습 데이터 정확도']:.3f}로 지문대로 학습한 결과와 다릅니다. 입력이 X_number_train인지, 학습 데이터 정확도를 학습 데이터 전체로 구했는지 보세요."
assert 0.05 < number_last["테스트 정확도"] < 0.25, f"마지막 테스트 정확도가 {number_last['테스트 정확도']:.3f}로 지문대로 학습한 결과와 다릅니다. 테스트 입력이 X_number_test인지, 테스트 정확도를 테스트 데이터로 구했는지 보세요."
print("1-3 확인 완료: 마지막 학습 손실", round(number_last["학습 손실"], 3), "| 테스트 정확도", round(number_last["테스트 정확도"], 3))

**결과 읽기**
- **20에폭 행의 `학습 손실` 1.946:** log(7) = 1.95와 거의 같아 학습 전과 다름없습니다.
- **`학습 데이터 정확도` 0.152, `테스트 정확도` 0.115:** 무작위로 고를 때의 정확도(약 0.14)와 같은 수준입니다.
- **원인:** 학습률 때문인지 다음 절에서 확인합니다.

## 2. 학습률을 바꿔 보기

**배경**: 손실이 log(7) 근처에 머문 원인이 학습률인지, 10배 간격 학습률 네 개로 다시 학습해 확인합니다.

**요구사항**
- 학습률 0.0001, 0.001, 0.01, 0.1마다 숫자 입력을 20에폭, `"adam"`으로 학습하세요.
- 학습률마다 `학습률`, `마지막 학습 손실`, `학습 데이터 정확도` 세 키의 dict를 만들어 표 **`lr_table`**(DataFrame, 4행)에 담으세요. 값은 기록 표의 마지막 행에서 꺼냅니다.

**예시**: 학습률 0.001 행은 1-3과 설정이 같아 `number_history`의 마지막 행과 값이 같습니다.

**확인 기준**: `lr_table`은 학습률이 작은 것부터 4행이고, 열은 `학습률`, `마지막 학습 손실`, `학습 데이터 정확도` 순서입니다. 학습률 0.001 행은 1-3의 마지막 에폭과 0.0003 이내로 같고, 네 행의 마지막 학습 손실은 서로 같지 않고(최댓값과 최솟값의 차가 0.002보다 큼), 모든 행의 마지막 학습 손실은 1.8을 넘으며 학습 데이터 정확도는 0.25 미만입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 반복문 한 번에 학습 한 번, lr_rows에 한 줄을 더합니다.

세부구현:
1. 숫자 입력 학습과 같은 인자로 학습하되 학습률만 반복문의 lr로 바꿉니다.
2. 기록 표의 마지막 행(last)에서 학습 손실 열과 학습 데이터 정확도 열의 값을 꺼냅니다.
3. dict의 키는 요구사항의 세 이름을 그대로 씁니다.
```

</details>

In [ ]:
# [작성] 학습률 네 개로 숫자 입력 모델을 학습해 표로 비교하세요.
lr_rows = []
for lr in [0.0001, 0.001, 0.01, 0.1]:
    # 1. 숫자 입력, 20에폭, adam으로 학습
    _, history = ...

    # 2. 마지막 에폭(맨 아래 행)의 값으로 한 행
    last = history.iloc[-1]
    lr_rows.append(...)

lr_table = pd.DataFrame(lr_rows)
display(lr_table.round({"마지막 학습 손실": 4, "학습 데이터 정확도": 4}))

In [ ]:
# [자가채점] 표 모양과 학습 결과를 확인합니다.
assert isinstance(lr_table, pd.DataFrame) and len(lr_table) == 4, "lr_table은 학습률마다 한 행, 4행 표입니다. `...`를 채웠는지 확인하세요."
assert list(lr_table.columns) == ["학습률", "마지막 학습 손실", "학습 데이터 정확도"], "lr_table의 열은 학습률, 마지막 학습 손실, 학습 데이터 정확도 순서입니다. 설정을 비교하는 표라 테스트 정확도는 넣지 않습니다."
assert list(lr_table["학습률"]) == [0.0001, 0.001, 0.01, 0.1], "행은 학습률 0.0001, 0.001, 0.01, 0.1 순서입니다."

row_001 = lr_table.iloc[1]
assert abs(row_001["마지막 학습 손실"] - number_history.iloc[-1]["학습 손실"]) < 3e-4, "학습률 0.001 행은 숫자 입력 학습과 같은 설정이라 값도 같아야 합니다. 숫자 입력, 20에폭, adam으로 학습했는지 보세요."
assert lr_table["마지막 학습 손실"].max() - lr_table["마지막 학습 손실"].min() > 0.002, "네 행의 마지막 학습 손실이 거의 같습니다. 반복문의 lr을 train_weekday_model에 넘겼는지 보세요. 학습률이 다르면 손실도 조금씩 달라집니다."
assert (lr_table["마지막 학습 손실"] > 1.8).all(), "마지막 학습 손실이 지문대로 학습한 결과와 다른 행이 있습니다. 입력이 X_number_train인지, 반복문의 lr을 train_weekday_model에 넘겼는지 보세요."
assert (lr_table["학습 데이터 정확도"] < 0.25).all(), "학습 데이터 정확도가 지문대로 학습한 결과와 다른 행이 있습니다. 기록 표의 학습 데이터 정확도 열을 꺼냈는지 보세요."
print("2 확인 완료")

**결과 읽기**
- **`마지막 학습 손실` 1.9454~1.9525:** 학습률을 1,000배 범위(0.0001~0.1)로 바꿔도 log(7) 근처에 머뭅니다.
- **`학습 데이터 정확도` 0.1436~0.1557:** 무작위로 고를 때의 수준입니다. 학습 데이터조차 맞히지 못하므로 손실이 줄지 않은 원인은 학습률이 아닙니다.

## 3. 원-핫 입력 만들기

**배경**: 학습률이 원인이 아니므로 입력을 원-핫 인코딩(값마다 칸을 하나씩 두고 그 값의 칸만 1, 나머지는 0으로 나타내는 방법)으로 바꿉니다.

**요구사항**
- `요일`을 뺀 `연`, `월`, `일` 세 열을 `pd.get_dummies`로 펼친 표 **`onehot_table`**(DataFrame, 10,958행 73열)을 만드세요. 열은 연 30칸(2000~2029), 월 12칸(1~12), 일 31칸(1~31) 순서입니다.
    - 회귀분석 단원에서와 달리 기준 범주를 빼지 않고 값마다 칸을 모두 만듭니다.
    - **10,958행 전체를 먼저 펼친 뒤** 나눕니다. 학습 행과 테스트 행을 따로 펼치면 6번처럼 연도로 나눌 때 학습 쪽은 연 25칸, 테스트 쪽은 연 5칸이 되어 모양이 맞지 않습니다.
- `onehot_table`을 float32 텐서 **`X_onehot`**(모양 `(10958, 73)`)으로 바꾸세요.
- `X_onehot`에서 학습 행만 고른 텐서를 **`X_onehot_train`** 에 담으세요.
- `X_onehot`에서 테스트 행만 고른 텐서를 **`X_onehot_test`** 에 담으세요.

**예시**: 첫 행(2000년 1월 1일)은 `연_2000`, `월_1`, `일_1` 세 칸만 1이고 나머지 70칸은 0입니다.

**확인 기준**: `onehot_table`이 `(10958, 73)` 모양의 DataFrame이고, `X_onehot`은 같은 모양의 float32 텐서입니다. 행마다 1이 세 개(연, 월, 일 하나씩)이며 1인 자리가 그 행의 날짜와 맞고, `X_onehot_train`, `X_onehot_test`는 `X_onehot`에서 각각 `train_index`, `test_index` 행을 고른 것입니다.

<details><summary>힌트</summary>

```text
접근방법:
- get_dummies 함수는 글자 열만 저절로 펼칩니다. 숫자 열인 연, 월, 일은 펼칠 열을 직접 알려 줘야 합니다.

세부구현:
1. dates에서 연, 월, 일 세 열만 고른 표를 get_dummies에 넣고, 펼칠 열을 지정하는 인자에 세 열 이름을 줍니다. 빼면 3칸 그대로 남습니다.
2. 0과 1의 자료형은 자료형을 지정하는 인자로 정합니다. 주지 않으면 True, False가 나옵니다.
3. 표를 넘파이 배열로 바꾼 뒤 float32 텐서로 만듭니다.
4. X_onehot에서 train_index 행과 test_index 행을 각각 고릅니다.
```

</details>

In [ ]:
# [작성] 연, 월, 일을 원-핫으로 펼친 입력을 만드세요.

# 1. 연, 월, 일 세 열을 원-핫 표로 펼치기(요일은 정답이라 넣지 않음)
onehot_table = ...
print("원-핫 표 모양:", onehot_table.shape)

# 첫 행에서 1인 칸의 이름
first_row = onehot_table.iloc[0]
print("첫 행에서 1인 칸:", list(onehot_table.columns[first_row == 1]))

# 2. float32 텐서
X_onehot = ...

# 3. 학습 입력과 테스트 입력
X_onehot_train = ...
X_onehot_test = ...

In [ ]:
# [자가채점] 원-핫 입력의 모양과, 1인 자리가 날짜와 맞는지 확인합니다.
assert isinstance(onehot_table, pd.DataFrame), "onehot_table은 pd.get_dummies가 돌려준 DataFrame입니다. `...`를 채웠는지 확인하세요."
assert onehot_table.shape == (len(dates), 73), f"onehot_table은 (10958, 73) 모양입니다. 지금은 {onehot_table.shape}입니다. 연, 월, 일 세 열만 골라 펼칠 열로 지정했는지 보세요."
assert isinstance(X_onehot, torch.Tensor) and X_onehot.dim() == 2 and len(X_onehot) == len(dates), "X_onehot은 날짜마다 한 행(10,958행)인 2차원 텐서입니다. `...`를 채웠는지, 전체 행을 펼쳤는지 확인하세요."
assert X_onehot.shape[1] == 73, f"X_onehot의 열은 연 30 + 월 12 + 일 31 = 73칸입니다. 지금은 {X_onehot.shape[1]}칸입니다. 3칸이면 펼칠 열을 지정하지 않은 것이고, 74칸이나 80칸이면 요일 열이 남은 것입니다."
assert X_onehot.dtype == torch.float32, "X_onehot은 float32 텐서로 만드세요. 텐서를 만들 때 자료형을 float32로 지정합니다."
assert torch.equal(X_onehot.sum(dim=1), torch.full((len(dates),), 3.0)), "원-핫 입력은 행마다 1이 정확히 세 개(연, 월, 일 하나씩)이고 나머지는 0입니다."

years = torch.tensor(dates["연"].to_numpy())
months = torch.tensor(dates["월"].to_numpy())
days = torch.tensor(dates["일"].to_numpy())
assert torch.equal(X_onehot[:, :30].argmax(dim=1), years - 2000), "앞 30칸은 연 2000~2029입니다. 열 순서가 연, 월, 일인지 보세요."
assert torch.equal(X_onehot[:, 30:42].argmax(dim=1), months - 1), "가운데 12칸은 월 1, 2, ..., 12 순서입니다. 숫자를 글자로 바꿔 펼치면 1, 10, 11, 12, 2 순서가 되니, 숫자 열 그대로 펼칠 열로 지정하세요."
assert torch.equal(X_onehot[:, 42:].argmax(dim=1), days - 1), "마지막 31칸은 일 1, 2, ..., 31 순서입니다. 숫자를 글자로 바꿔 펼치면 순서가 바뀝니다."
assert torch.equal(X_onehot_train, X_onehot[train_index]), "X_onehot_train은 X_onehot에서 train_index 행을 고른 것입니다."
assert torch.equal(X_onehot_test, X_onehot[test_index]), "X_onehot_test는 X_onehot에서 test_index 행을 고른 것입니다."
print("3 확인 완료")

## 4. 원-핫 입력으로 학습하기

**배경**: 1-3과 모델 구조, 학습 설정을 모두 같게 두고 입력만 원-핫으로 바꿔, 입력 표현이 결과를 얼마나 바꾸는지 확인합니다.

**요구사항**
- 원-핫 입력(`X_onehot_train`, `X_onehot_test`)을 학습률 0.001, 20에폭, `"adam"`으로 학습하세요.
- 학습한 모델을 **`onehot_model`**(nn.Sequential)에 담으세요.
- 기록 표를 **`onehot_history`**(DataFrame)에 담으세요.

**확인 기준**: `onehot_history`는 20행이고 `onehot_model`의 첫 선형 층 입력 크기가 73입니다. 마지막 에폭의 학습 데이터 정확도와 테스트 정확도가 모두 0.92를 넘습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 숫자 입력 학습의 호출에서 입력 두 개만 원-핫 입력으로 바꿉니다.
```

</details>

In [ ]:
# [작성] 원-핫 입력으로 20에폭 학습하세요.
onehot_model, onehot_history = ...
display(onehot_history.round(3))

In [ ]:
# [자가채점] 원-핫 입력으로 학습한 결과를 확인합니다.
assert isinstance(onehot_history, pd.DataFrame) and len(onehot_history) == 20, "onehot_history는 20에폭의 기록 표(20행)입니다. `...`를 채웠는지 확인하세요."
assert onehot_model[0].in_features == 73, "원-핫 입력이면 첫 선형 층의 입력 크기가 73입니다. X_onehot_train을 넘겼는지 보세요."

onehot_last = onehot_history.iloc[-1]
assert onehot_last["학습 데이터 정확도"] > 0.92, f"마지막 학습 데이터 정확도가 {onehot_last['학습 데이터 정확도']:.3f}로 지문대로 학습한 결과(0.92 초과)와 다릅니다. 함수 안에서 train_one_epoch로 실제로 학습하는지 보세요. 학습용 DataLoader가 학습 데이터를 섞는지도 보세요."
assert onehot_last["테스트 정확도"] > 0.92, f"마지막 테스트 정확도가 {onehot_last['테스트 정확도']:.3f}로 지문대로 학습한 결과(0.92 초과)와 다릅니다. 테스트 입력이 X_onehot_test인지 보세요. 학습용 DataLoader가 학습 데이터를 섞는지도 보세요."
print("4 확인 완료: 마지막 테스트 정확도", round(onehot_last["테스트 정확도"], 3))

아래 셀은 실행만 하세요. 두 입력의 에폭별 학습 손실과 테스트 정확도를 나란히 그립니다. 회색 점선은 무작위로 고를 때의 값(손실 log(7), 정확도 1/7)입니다.

In [ ]:
import math

# 숫자 입력과 원-핫 입력의 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for history, label in [(number_history, "숫자 입력(3칸)"), (onehot_history, "원-핫 입력(73칸)")]:
    axes[0].plot(history["에폭"], history["학습 손실"], marker="o", label=label)
    axes[1].plot(history["에폭"], history["테스트 정확도"], marker="o", label=label)

# 무작위로 고를 때의 기준선
axes[0].axhline(math.log(7), color="gray", linestyle="--", label="log(7)")
axes[1].axhline(1 / 7, color="gray", linestyle="--", label="1/7")

axes[0].set_title("에폭별 학습 손실", fontsize=14)
axes[1].set_title("에폭별 테스트 정확도", fontsize=14)
for ax in axes:
    ax.set_xlabel("에폭", fontsize=12)
    ax.set_xticks(range(2, 21, 2))
    ax.tick_params(labelsize=11)
    ax.legend(loc="center right", fontsize=11)  # 두 선이 위아래로 갈라지는 가운데 오른쪽 빈자리에 둡니다
plt.tight_layout()
plt.show()

**결과 읽기**
- **`숫자 입력(3칸)` 선, 손실 log(7)과 테스트 정확도 1/7 근처:** 20에폭 내내 머뭅니다. 요일은 7일마다 처음으로 돌아와 값의 크기와 관계가 없으므로, 크기를 그대로 받는 입력으로는 배우지 못했습니다.
- **`원-핫 입력(73칸)`, 테스트 정확도 0.99:** 값마다 칸과 가중치가 따로 있어 크기 순서를 가정하지 않습니다. 처음 4에폭은 기준선 근처에 머물다가 5에폭부터 손실이 줄어 20에폭에는 0.99가 됩니다.

## 5. 옵티마이저와 학습률 비교하기

**배경**: 원-핫 입력도 처음 4에폭은 손실이 log(7) 근처에 머물렀으므로, 5에폭 안에 이 구간을 벗어나는 옵티마이저와 학습률을 학습 손실로 비교합니다.

**요구사항**
- 아래 여섯 설정마다 원-핫 입력을 5에폭 학습하세요(표 순서 그대로).

| 옵티마이저 이름 | 학습률 |
|---|---|
| `"sgd"` | 0.1 |
| `"momentum"` | 0.1 |
| `"adam"` | 0.001 |
| `"adam"` | 0.003 |
| `"adam"` | 0.01 |
| `"adam"` | 0.03 |

- 설정마다 `옵티마이저`, `학습률`, `마지막 학습 손실` 세 키의 dict를 만들어 표 **`optimizer_table`**(DataFrame, 6행)에 담으세요. `옵티마이저` 값은 위 표의 옵티마이저 이름입니다.
- 설정마다 기록 표를 **`optimizer_histories`**(dict)에 모으는 줄은 골격에 있습니다(아래 그래프가 씁니다). 그대로 두세요.

**예시**: 첫 행은 `{"옵티마이저": "sgd", "학습률": 0.1, "마지막 학습 손실": 1.95}`처럼 생겼습니다(손실 값은 예).

**확인 기준**: `optimizer_table`은 위 표 순서의 6행이고 열은 `옵티마이저`, `학습률`, `마지막 학습 손실` 순서이며, `optimizer_histories`의 기록 표는 설정마다 5행입니다. `"sgd"`와 `"momentum"`(학습률 0.1)의 마지막 학습 손실은 1.9를 넘고, `"adam"` 0.003은 1.0 미만입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 2절 반복문과 같은 모양입니다. 입력은 원-핫, 에폭 수는 5, 옵티마이저 이름과 학습률은 반복문의 두 변수입니다.

세부구현:
1. 반복문의 optimizer_name과 lr을 train_weekday_model의 같은 이름 인자로 넘깁니다.
2. dict의 옵티마이저 값은 반복문의 optimizer_name이고, 마지막 학습 손실은 last의 학습 손실 열입니다.
```

</details>

In [ ]:
# [작성] 여섯 설정으로 원-핫 입력을 5에폭씩 학습해 표로 비교하세요.
optimizer_rows = []
optimizer_histories = {}  # 설정 이름 -> 기록 표(아래 그래프가 씁니다)
for optimizer_name, lr in [("sgd", 0.1), ("momentum", 0.1), ("adam", 0.001), ("adam", 0.003), ("adam", 0.01), ("adam", 0.03)]:
    # 1. 원-핫 입력, 5에폭으로 학습
    _, history = ...
    optimizer_histories[f"{optimizer_name} {lr}"] = history

    # 2. 마지막 에폭의 학습 손실로 한 행
    last = history.iloc[-1]
    optimizer_rows.append(...)

optimizer_table = pd.DataFrame(optimizer_rows)
display(optimizer_table.round(3))

In [ ]:
# [자가채점] 표 모양과 학습 결과를 확인합니다.
assert isinstance(optimizer_table, pd.DataFrame) and len(optimizer_table) == 6, "optimizer_table은 설정마다 한 행, 6행 표입니다. `...`를 채웠는지 확인하세요."
assert list(optimizer_table.columns) == ["옵티마이저", "학습률", "마지막 학습 손실"], "optimizer_table의 열은 옵티마이저, 학습률, 마지막 학습 손실 순서입니다."
assert list(optimizer_table["옵티마이저"]) == ["sgd", "momentum", "adam", "adam", "adam", "adam"], "행은 지문 표의 순서(sgd, momentum, adam 네 개)입니다."
assert list(optimizer_table["학습률"]) == [0.1, 0.1, 0.001, 0.003, 0.01, 0.03], "학습률은 지문 표의 순서(0.1, 0.1, 0.001, 0.003, 0.01, 0.03)입니다."
assert len(optimizer_histories["adam 0.01"]) == 5, "에폭 수는 5입니다."

losses = list(optimizer_table["마지막 학습 손실"])
assert losses[0] > 1.9 and losses[1] > 1.9, f"SGD, 모멘텀(0.1)의 마지막 학습 손실 {losses[0]:.3f}, {losses[1]:.3f}가 지문대로 학습한 결과와 다릅니다. 입력이 X_onehot_train인지, 반복문의 옵티마이저 이름과 학습률을 넘겼는지, 에폭 수가 5인지 보세요."
assert losses[3] < 1.0, f"Adam 0.003의 마지막 학습 손실 {losses[3]:.3f}가 지문대로 학습한 결과와 다릅니다. 입력이 X_onehot_train인지, train_weekday_model의 adam에 학습률을 줬는지, 에폭 수가 5인지 보세요."
print("5 확인 완료")

아래 셀은 실행만 하세요. 설정마다 에폭별 학습 손실을 그립니다. 선이 빨리 내려갈수록 같은 에폭 안에 더 많이 배운 설정입니다.

In [ ]:
# 설정마다 에폭별 학습 손실
fig, ax = plt.subplots(figsize=(9, 4.5))
# log(7) 근처에서 겹치는 세 설정은 선 모양, 속이 빈 마커, 마커 크기를 달리해 겹쳐도 셋 다 보이게 합니다
line_styles = {"sgd 0.1": ("--", "s", 14), "momentum 0.1": (":", "^", 10), "adam 0.03": ("-.", "o", 6)}
for setting, history in optimizer_histories.items():
    if setting in line_styles:
        linestyle, marker, size = line_styles[setting]
        ax.plot(history["에폭"], history["학습 손실"], linestyle=linestyle, marker=marker, markersize=size,
                markerfacecolor="none", markeredgewidth=1.5, label=setting)
    else:
        ax.plot(history["에폭"], history["학습 손실"], marker="o", label=setting)
ax.axhline(math.log(7), color="gray", linestyle="--", linewidth=1, zorder=0, label="log(7)")
ax.set_title("옵티마이저와 학습률별 학습 손실(원-핫 입력, 5에폭)")
ax.set_xlabel("에폭")
ax.set_ylabel("학습 손실")
ax.set_xticks(range(1, 6))
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))  # 범례가 선을 가리지 않게 그래프 오른쪽 밖에 둡니다
plt.tight_layout()
plt.show()

**결과 읽기**
- **`sgd 0.1`, `momentum 0.1`, `adam 0.03`의 `마지막 학습 손실` 1.946, 1.947, 1.947:** log(7) = 1.95에서 거의 줄지 않아, 그래프에서 세 선이 log(7) 점선 위에 겹칩니다. Adam도 학습률이 너무 크면(0.03) 줄지 않습니다.
- **Adam 0.001 1.843, 0.003 0.300, 0.01 0.048:** 0.001은 정체하고, 0.003과 0.01은 수렴 기준(손실 1.0 아래)을 넘었습니다.
- **0.003 대 0.01:** 둘 다 수렴 기준을 넘었으므로 손실이 더 낮은 쪽이 아니라 교안 01의 규칙으로 고릅니다.
- **AdamW:** 실무에서는 기본으로 쓰지만, 교안 02에서 Adam과 AdamW의 5에폭 학습 손실이 거의 같았으므로(0.01에서 Adam 0.1025, AdamW 0.0991) 이 과제는 Adam만 비교합니다.

### 5-1. 설정 고르기

**배경**: 교안 01의 학습률 규칙(1. 10배 간격으로 넓게 시도, 2. 발산 경계 사이 값을 더 시도, 3. 정한 횟수 안에 수렴한 후보 가운데 발산 경계에서 가장 먼 값)으로 위 여섯 설정 가운데 하나를 고릅니다. 5절의 학습률은 10배 간격 후보(0.001, 0.01)와 그 사이 값(0.003, 0.03)이라 2단계까지 시도한 셈입니다. 이 과제에서는 학습률이 커도 손실이 커지지 않고 log(7) 근처에 머물므로, 큰 쪽에서 수렴하지 못하기 시작하는 값(Adam 0.03)을 발산 경계 대신 씁니다.

**요구사항**
- 설정은 학습 손실로만 고릅니다. 테스트 정확도로 고르면 그 값이 처음 보는 데이터의 성능보다 좋게 나옵니다.
- (1) **수렴한 후보**는 **수렴 기준(손실 1.0 아래)** 을 만족한 설정, 곧 5에폭 뒤 마지막 학습 손실이 1.0 아래인 설정입니다. `optimizer_table`에서 이 행만 남긴 표를 **`converged_table`**(DataFrame, 열은 `optimizer_table`과 같음)에 담으세요.
- (2) 수렴한 후보는 모두 경계(Adam 0.03)보다 학습률이 작으므로, 경계에서 가장 먼 것은 **학습률이 가장 작은 후보**입니다. `converged_table`에서 이 행을 **`best_row`**(Series)에 담으세요.

**확인 기준**: `converged_table`이 `optimizer_table`과 같은 열이고, 마지막 학습 손실이 1.0 아래인 행만 정확히 2개 있습니다. `best_row`는 `converged_table`의 행이며 그 가운데 학습률이 가장 작은 행입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 표에서 수렴한 행만 남기고, 남은 행 가운데 학습률이 가장 작은 행을 꺼냅니다.

세부구현:
1. 표에서 조건에 맞는 행만 남길 때는 표 이름 뒤 대괄호 안에 조건을 씁니다. 조건은 마지막 학습 손실 열이 1.0보다 작은 것입니다.
2. 남은 표를 학습률 열 기준으로 오름차순 정렬합니다.
3. 정렬한 표의 맨 위 행을 위치로 꺼냅니다.
```

</details>

In [ ]:
# [작성] 수렴한 후보 가운데 발산 경계에서 가장 먼 설정을 고르세요.

# 1. 5에폭 뒤 마지막 학습 손실이 1.0 아래인 설정만 남기기
converged_table = ...
display(converged_table.round(3))

# 2. 남은 설정 가운데 학습률이 가장 작은 행
best_row = ...
print("고른 설정:", best_row["옵티마이저"], best_row["학습률"], "| 마지막 학습 손실:", round(best_row["마지막 학습 손실"], 4))

In [ ]:
# [자가채점] 수렴한 후보와 고른 설정을 확인합니다.

# 수렴한 후보: 남은 행은 모두 마지막 학습 손실이 1.0 아래이고, 지문대로 학습하면 2개입니다
assert isinstance(converged_table, pd.DataFrame), "converged_table은 optimizer_table에서 행을 골라 남긴 DataFrame입니다. `...`를 채웠는지 확인하세요."
assert list(converged_table.columns) == list(optimizer_table.columns), "converged_table의 열은 optimizer_table과 같습니다. 행만 고르고 열은 그대로 두세요."
assert (converged_table["마지막 학습 손실"] < 1.0).all(), "converged_table에 마지막 학습 손실이 1.0 이상인 설정이 남아 있습니다. 수렴한 후보의 기준을 다시 보세요."
assert len(converged_table) == 2, f"converged_table에 설정이 {len(converged_table)}개입니다. 지문대로 학습하면 마지막 학습 손실이 1.0 아래인 설정은 2개입니다. 기준이 1.0이고 '아래'(작다)인지 보세요."

# best_row: 수렴한 후보 가운데 하나이고, 남은 후보 어느 것보다도 큰 학습률 쪽 경계(0.03)에서 멀어야 합니다
assert isinstance(best_row, pd.Series), "best_row는 converged_table의 한 행(Series)입니다. 표에서 행 하나를 꺼내세요."
assert best_row["마지막 학습 손실"] < 1.0, f"고른 설정의 마지막 학습 손실이 {best_row['마지막 학습 손실']:.3f}입니다. optimizer_table이 아니라 수렴한 후보만 남긴 converged_table에서 고르세요."
for other_lr in converged_table["학습률"].tolist():
    assert best_row["학습률"] <= other_lr, f"고른 학습률 {best_row['학습률']}보다 큰 학습률 쪽 경계(0.03)에서 더 먼 수렴한 후보({other_lr})가 있습니다. 손실이 가장 낮은 설정이 아니라, 수렴한 후보 가운데 경계에서 가장 먼 설정인지 다시 보세요."
print("5-1 확인 완료")

## 6. 처음 보는 연도 맞히기

**배경**: 4번의 테스트 날짜는 같은 해의 다른 날짜가 학습 데이터에 있었으므로, 이번에는 2024년까지만 학습하고 2025~2029년을 맞혀 학습에 없던 연도에도 원-핫 입력이 통하는지 확인합니다.

**요구사항**
- `연`이 2024 이하인 행 번호를 정수 텐서 **`past_index`** 에 담으세요.
- `연`이 2025 이상인 행 번호를 정수 텐서 **`future_index`** 에 담으세요.
- `X_onehot`과 `y_all`의 `past_index` 행으로 학습하고 `future_index` 행으로 테스트하세요(학습률 0.001, 20에폭, `"adam"`).
- 학습한 모델을 **`future_model`**(nn.Sequential)에 담으세요.
- 기록 표를 **`future_history`**(DataFrame)에 담으세요.

**예시**: 2024년 12월 31일은 `past_index`, 2025년 1월 1일은 `future_index`에 들어갑니다. `past_index`는 9,132개, `future_index`는 1,826개입니다.

**확인 기준**: `past_index`와 `future_index`를 합치면 10,958개이고, 학습 행에 2025년 이후가 하나도 없으며, `future_history`는 20행이고 `future_model`의 첫 선형 층 입력 크기가 73입니다. 마지막 에폭의 학습 데이터 정확도가 0.92를 넘고 테스트 정확도가 0.3 미만입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 연도 조건으로 행 번호를 둘로 나누고, 그 행 번호로 입력과 정답을 골라 train_weekday_model에 넘깁니다.

세부구현:
1. dates의 연 열로 2024 이하 조건을 만들고, dates의 행 번호(index)를 그 조건으로 걸러 행 번호를 얻습니다.
2. 1의 행 번호를 정수 텐서로 바꿉니다. 2025 이상도 같은 방법으로 만듭니다.
3. X_onehot과 y_all에서 past_index 행을 고르면 학습 입력과 정답이고, future_index 행을 고르면 테스트 입력과 정답입니다.
4. train_weekday_model에 3의 네 텐서를 넘깁니다.
```

</details>

In [ ]:
# [작성] 2024년까지로 학습하고 2025~2029년으로 테스트하세요.

# 1. 2024년까지인 행 번호와 2025년부터인 행 번호
past_index = ...
future_index = ...
print("학습 행:", len(past_index), "| 테스트 행:", len(future_index))

# 2. 과거 행으로 학습하고 미래 행으로 테스트
future_model, future_history = ...
display(future_history.round(3))

In [ ]:
# [자가채점] 연도로 나눈 행 번호와, 처음 보는 연도의 결과를 확인합니다.
assert isinstance(past_index, torch.Tensor) and isinstance(future_index, torch.Tensor), "past_index, future_index는 행 번호를 담은 정수 텐서입니다. `...`를 채웠는지 확인하세요."
years = torch.tensor(dates["연"].to_numpy())
assert len(past_index) + len(future_index) == len(dates), "past_index와 future_index를 합치면 모든 행(10,958개)입니다."
assert years[past_index].max().item() == 2024, "past_index에는 2024년까지만 들어갑니다. 2025년 이후 행이 학습에 섞이면 처음 보는 연도가 아니게 됩니다."
assert years[future_index].min().item() == 2025, "future_index에는 2025년부터 들어갑니다."
assert isinstance(future_history, pd.DataFrame) and len(future_history) == 20, "future_history는 20에폭의 기록 표(20행)입니다."
assert isinstance(future_model, nn.Sequential) and future_model[0].in_features == 73, "future_model은 원-핫 입력(73칸)으로 학습한 모델입니다. 학습 입력이 past_index 행의 X_onehot인지 보세요."

future_last = future_history.iloc[-1]
assert future_last["학습 데이터 정확도"] > 0.92, f"마지막 학습 데이터 정확도가 {future_last['학습 데이터 정확도']:.3f}로 지문대로 학습한 결과(0.92 초과)와 다릅니다. 학습 입력이 past_index 행의 X_onehot인지 보세요. 학습용 DataLoader가 학습 데이터를 섞는지도 보세요."
assert future_last["테스트 정확도"] < 0.3, f"마지막 테스트 정확도가 {future_last['테스트 정확도']:.3f}로 지문대로 학습한 결과(0.3 미만)와 다릅니다. 테스트에 future_index 행을 넘겼는지, 학습에 2025년 이후 행이 섞이지 않았는지 보세요. 학습용 DataLoader가 학습 데이터를 섞는지도 보세요."
print("6 확인 완료: 학습 데이터 정확도", round(future_last["학습 데이터 정확도"], 3), "| 테스트 정확도", round(future_last["테스트 정확도"], 3))

아래 셀은 실행만 하세요. 무작위로 나눈 실험(테스트 날짜의 연도를 학습에서 봄)과 연도로 나눈 실험(테스트 연도를 학습에서 못 봄)의 마지막 에폭 정확도를 막대로 비교합니다.

In [ ]:
# 무작위로 나눈 실험과 연도로 나눈 실험의 마지막 에폭 정확도
split_names = ["무작위로 나눔", "연도로 나눔"]
train_accuracies = [onehot_history["학습 데이터 정확도"].iloc[-1], future_history["학습 데이터 정확도"].iloc[-1]]
test_accuracies = [onehot_history["테스트 정확도"].iloc[-1], future_history["테스트 정확도"].iloc[-1]]

# 학습 데이터 정확도와 테스트 정확도 막대를 나란히 놓으려고 x 위치를 0.2씩 비킵니다
fig, ax = plt.subplots(figsize=(7, 4.5))
positions = range(len(split_names))
ax.bar([p - 0.2 for p in positions], train_accuracies, width=0.4, label="학습 데이터 정확도")
ax.bar([p + 0.2 for p in positions], test_accuracies, width=0.4, label="테스트 정확도")
ax.axhline(1 / 7, color="gray", linestyle="--", label="1/7")

# 막대 위에 값 적기(1/7 점선과 겹치지 않게 막대 끝에서 0.05 위)
for p in positions:
    ax.text(p - 0.2, train_accuracies[p] + 0.05, f"{train_accuracies[p]:.2f}", ha="center")
    ax.text(p + 0.2, test_accuracies[p] + 0.05, f"{test_accuracies[p]:.2f}", ha="center")

ax.set_xticks(list(positions))
ax.set_xticklabels(split_names)
ax.set_ylim(0, 1.15)
ax.set_title("원-핫 입력의 마지막 에폭 정확도")
ax.legend(loc="center right")
plt.tight_layout()
plt.show()

**결과 읽기**
- **테스트 정확도 0.99 대 0.10:** 무작위로 나눈 실험은 0.99인데, 연도로 나눈 실험은 학습 데이터 정확도 1.00에 테스트 정확도 0.10으로 무작위로 고를 때의 정확도(약 0.14)와 비슷합니다.
- **2025~2029년 칸:** 학습 데이터에서 늘 0입니다. 그 칸에 곱해지는 가중치는 기울기도 늘 0이라 한 번도 갱신되지 않고, 모델은 처음 보는 연도가 요일을 며칠 미는지 모릅니다. 원-핫 입력은 학습에서 본 값만 배웁니다.